In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

import os # 운영체제 관련 기능

from pathlib import Path

BASE_DIR = Path.cwd().parent # 현재 스크립트의 상위 디렉토리: bareun/

# =========================
# 1) 데이터 읽기
# =========================
CSV_PATH = BASE_DIR / "csv" / "통계용" / "2025-12-26_01-39_logodds_V_No_by_ASP_No_ID.csv"
df = pd.read_csv(CSV_PATH)

# =========================
# 2) 분석에 쓸 조건(필터)
#    - ASP_No는 지정한 12개만
#    - outcome_level은 1000 이상
#    - 2999, 1009, 4999 제외
# =========================
ASP_SET = [101, 102, 111, 112, 123, 122, 124, 113, 125, 126, 132, 131]
EXCLUDE = {2999, 1009, 4999}

df = df[df["ASP_No"].isin(ASP_SET)].copy()
df = df[df["outcome_level"] >= 1000].copy()
df = df[~df["outcome_level"].isin(EXCLUDE)].copy()

# =========================
# 3) long → wide로 바꾸기 (feature matrix 만들기)
#    - 행: outcome_level
#    - 열: ASP_No(12개)
#    - 값: log_odds
#    - 없는 조합은 0으로 채움(=중립)
# =========================
X = (
    df.pivot(
        index="outcome_level", 
        columns="ASP_No", 
        values="log_odds",
        aggfunc="mean"          # if duplicates, average them    
    )
      .reindex(columns=ASP_SET)   # 열 순서 고정
      .fillna(0.0)
)

print("X shape:", X.shape)  # (outcome_level 개수, 12)

# =========================
# 4) 스케일링(표준화)
#    - ASP별 값의 스케일 차이가 군집에 과영향 주는 걸 줄이려고 함
# =========================
scaler = StandardScaler() # 스케일러 객체 생성: 평균0, 분산1로 변환
X_scaled = scaler.fit_transform(X.values) # 스케일링 적용: numpy array 반환

# =========================
# 5) k-means 실행
#    - k: 군집 수 (네가 먼저 정해줘야 함)
#    - random_state: seed(재현성)
#    - n_init: 서로 다른 초기화로 여러 번 시도 후 "좋은 것"을 내부적으로 선택
# =========================
k = 2  # <-- 여기만 바꿔가면서 보면 됨(예: 3~12)
BASE_RANDOM_STATE = 42 # 재현성을 위한 시드 고정, 아무 숫자나 가능, 고정 안 하면 매번 결과 달라짐
N_INIT = 50 # 서로 다른 초기화 시도 횟수, 일반적으로 10 이상 권장, 10~100, 너무 크면 시간 오래 걸림, 보통 50 정도면 충분

model = KMeans(n_clusters=k, random_state=BASE_RANDOM_STATE, n_init=N_INIT) # k-means 모델 객체 생성: k 지정, seed 고정, 여러 번 시도
labels = model.fit_predict(X_scaled) # k-means 실행 및 라벨(군집번호) 할당: numpy array 반환

# =========================
# 6) 결과 확인
#    - silhouette: "뭉침/분리" 정도(내부평가 지표)
# =========================
# silhouette 점수 계산 sil= silhouette_score(X_scaled, labels) 
# 계산방법: 평균 실루엣 계수, 각 샘플의 실루엣 계수는 (b - a) / max(a, b)
#   - a: 같은 클러스터 내의 다른 샘플들과의 평균 거리
#   - b: 가장 가까운 다른 클러스터의 샘플들과의 평균 거리  

sil = silhouette_score(X_scaled, labels) # 실루엣 점수 계산: 클러스터링 품질 평가
                                         #: 값이 클수록 좋음: -1~1: 0 이상이면 군집화 의미 있음
print(f"k={k} silhouette={sil:.4f}")
print("inertia=", model.inertia_)

# =========================
# 7) 라벨(군집번호)을 outcome_level에 붙여서 저장/merge 가능하게 만들기
# =========================
cluster_df = pd.DataFrame({
    "outcome_level": X.index,
    "cluster": labels
})

cluster_df.head()

X shape: (1484, 12)
k=2 silhouette=0.2818
inertia= 12994.627371571569


,outcome_level,cluster
0,1001,1
1,2001,1
2,2002,1
3,2003,1
4,2004,1


In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# =========================================================
# 0) Load
# =========================================================
CSV_PATH = "/mnt/data/2025-12-26_01-39_logodds_V_No_by_ASP_No_ID.csv"
df = pd.read_csv(CSV_PATH)

# =========================================================
# 1) Filters (same as before)
# =========================================================
ASP_SET = [101, 102, 111, 112, 123, 122, 124, 113, 125, 126, 132, 131]
EXCLUDE = {2999, 1009, 4999}

df = df[df["ASP_No"].isin(ASP_SET)].copy()
df = df[df["outcome_level"] >= 1000].copy()
df = df[~df["outcome_level"].isin(EXCLUDE)].copy()

# =========================================================
# 2) Build feature matrix X (outcome_level x 12 ASP features)
#    - Using pivot_table (safer than pivot if duplicates exist)
#    - Missing combinations -> 0.0
# =========================================================
X = (
    df.pivot_table(
        index="outcome_level",
        columns="ASP_No",
        values="log_odds",
        aggfunc="mean"          # if duplicates, average them
    )
    .reindex(columns=ASP_SET)  # keep fixed column order
    .fillna(0.0)
)

print("X shape:", X.shape)  # (n_outcome_levels, 12)

# =========================================================
# 3) Scale features (recommended for K-means)
# =========================================================
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X.values)

# =========================================================
# 4) Try multiple k and pick the best by silhouette
# =========================================================
K_MIN = 2
K_MAX = 20

RANDOM_STATE = 42   # seed (fixed for reproducibility)
N_INIT = 20         # internal multiple initializations per k

# Optional: avoid k where any cluster becomes too tiny
MIN_CLUSTER_SIZE = None   # e.g., 20 (set None to disable)

rows = []
best_k = None
best_sil = -np.inf

for k in range(K_MIN, K_MAX + 1):
    if k >= X_scaled.shape[0]:
        continue  # cannot have more clusters than samples

    model = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=N_INIT) # k-means model 객체 생성
    labels = model.fit_predict(X_scaled) # k-means 실행, 군집화 결과 라벨 할당

    # Optional size check (skip unstable solutions with tiny clusters)
    if MIN_CLUSTER_SIZE is not None:
        sizes = np.bincount(labels) # 각 클러스터의 크기 계산
        if sizes.min() < MIN_CLUSTER_SIZE:
            rows.append({
                "k": k,
                "silhouette": np.nan, # skipped, so NaN
                "inertia": float(model.inertia_), # skipped, but still report inertia
                "min_cluster_size": int(sizes.min()), # skipped, but still report min size
                "note": "skipped (tiny cluster)" # skipped due to tiny cluster
            })
            continue

    sil = silhouette_score(X_scaled, labels) # silhouette 점수 계산

    rows.append({
        "k": k,
        "silhouette": float(sil),
        "inertia": float(model.inertia_),
        "min_cluster_size": int(np.bincount(labels).min()),
        "note": ""
    })

    if sil > best_sil:
        best_sil = sil
        best_k = k

scores_df = pd.DataFrame(rows).sort_values("k").reset_index(drop=True)
print(scores_df)

print("\nBest k by silhouette =", best_k, " (silhouette =", best_sil, ")")

# =========================================================
# 5) Fit final model with best_k and create outputs for merge/analysis
# =========================================================
final_model = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=N_INIT)
final_labels = final_model.fit_predict(X_scaled)

# outcome_level -> cluster label (this is what you merge back)
cluster_df = pd.DataFrame({
    "outcome_level": X.index,
    "cluster": final_labels
})

# Cluster profile: mean log_odds vector per cluster (interpretable summary)
X_with_cluster = X.copy()
X_with_cluster["cluster"] = final_labels
cluster_profile = X_with_cluster.groupby("cluster")[ASP_SET].mean()
cluster_sizes = X_with_cluster["cluster"].value_counts().sort_index()

print("\nCluster sizes:")
print(cluster_sizes)

print("\nCluster profile (mean log_odds per ASP):")
print(cluster_profile)

# If you want to save minimal outputs:
# scores_df.to_csv("kmeans_k_search_scores.csv", index=False, encoding="utf-8-sig")
# cluster_df.to_csv(f"kmeans_labels_bestk_{best_k}.csv", index=False, encoding="utf-8-sig")
# cluster_profile.to_csv(f"kmeans_profile_bestk_{best_k}.csv", encoding="utf-8-sig")


WindowsPath('c:/Users/yu2hy/OneDrive/◎2020_copus/python_2023/bareun/scrips_2025')